In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
from brain_image.prior import BrainDiffusionPrior
import torch
from brain_image.model import NICEConfig, ResidualAdapter
from brain_image.utils import get_mean_gradients

device = torch.device("cuda")

eeg_prior = BrainDiffusionPrior.from_pretrained().to(device)
cond_adapter = ResidualAdapter(latent_dim=768, hidden_factor=2).to(device)
x = torch.randn(3, 768, device=device)

times = torch.randint(0, 1000, (3,), device = device, dtype = torch.long)

eeg_prior.train()
eeg_prior.requires_grad_(True)
cond_adapter.train()
cond_adapter.requires_grad_(True)

x = x / x.norm(dim=-1, keepdim=True)
xp = cond_adapter(x)

loss, pred = eeg_prior(brain_embedding=xp, image_embedding=x, times=times)
loss.backward()

eeg_prior_grad = get_mean_gradients(eeg_prior)
cond_adapter_grad = get_mean_gradients(cond_adapter)

print("eeg_prior_grad", eeg_prior_grad)
print("cond_adapter_grad", cond_adapter_grad)

loss2, pred2 = eeg_prior(brain_embedding=xp + torch.randn_like(xp), image_embedding=x, times=times)

print("Did the pred change?", not torch.allclose(pred, pred2))

eeg_prior_grad tensor(0.0793, device='cuda:0')
cond_adapter_grad tensor(0.0037, device='cuda:0')
Did the pred change? True
